# Khmer OCR — from-scratch trainingRun cells top to bottom. **Safe to re-run after a Colab disconnect** — trainingresumes from the last checkpoint automatically.Order: setup → audit (Phase 0) → pack (Phase 1) → CRNN baseline (Phase 2) → AttnOCR (Phase 3)

## 0. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || print("NO GPU — Runtime > Change runtime type > T4")

In [ ]:
from google.colab import drivedrive.mount('/content/gdrive')import osROOT = '/content/gdrive/MyDrive/KhmerOCR'os.makedirs(f'{ROOT}/shards', exist_ok=True)      # LMDB shards (big files — Drive is fine)os.makedirs(f'{ROOT}/checkpoints', exist_ok=True) # survives disconnectsos.makedirs(f'{ROOT}/audit', exist_ok=True)print('ok')

In [ ]:
!pip install -q lmdb datasets huggingface_hub wandb# Source code lives in Drive so edits persist across sessions.CODE = f'{ROOT}/src'!mkdir -p {CODE}import sys; sys.path.insert(0, CODE)print('put normalize.py audit.py pack.py dataset.py models.py train.py in', CODE)

## 1. Phase 0 — Audit (CPU, ~10 min)**Do not skip this.** Most from-scratch OCR failures are label-encoding bugs.This finds them now instead of after a 10-hour run.**Gate:** kept charset should be **80–140**. Much higher means normalization isincomplete and the model will be asked to learn invisible distinctions.

In [ ]:
!cd {CODE} && python audit.py --out {ROOT}/audit --limit 5000 --sources \    SoyVitou/KhmerSynthetic1M \    KimkosalYon/khmer-ocr-200k-siemreap-arial \    Sokheng/khmer-synthetic-ocr-v1-100k \    SoyVitou/62k-images-khmer-printed-dataset \    Darayut/khmer-scene-text-synthetic-contrast \    Darayut/khmer-document-synthetic-low-res \    SoyVitou/khmer-handwritten-dataset-4.2k \    SoyVitou/KhmerST-Dataset-OCR-Cropped

In [ ]:
import jsoncs = json.load(open(f'{ROOT}/audit/charset.json'))print('charset size:', cs['size'])print(''.join(cs['charset']))print()print(open(f'{ROOT}/audit/suspects.csv').read()[:1500])

## 2. Phase 1 — Pack to LMDB (~2–4h, one time)Packing goes to **local `/content` disk**, not Drive. Reading 1M small filesthrough Drive's FUSE layer costs 10–50ms each — hours per epoch with the GPUidle. Shards are copied to Drive once at the end.**The two real datasets are held out as test data.** They are the onlyevidence that synthetic→real transfer worked, so they are never trained on.

In [ ]:
SYNTH = [    ('SoyVitou/KhmerSynthetic1M', 'synth1m'),    ('KimkosalYon/khmer-ocr-200k-siemreap-arial', 'arial200k'),    ('Sokheng/khmer-synthetic-ocr-v1-100k', 'synth100k'),    ('SoyVitou/62k-images-khmer-printed-dataset', 'printed62k'),    ('Darayut/khmer-scene-text-synthetic-contrast', 'scene'),    ('Darayut/khmer-document-synthetic-low-res', 'doclowres'),]REAL = [   # HELD OUT — test only, never trained on    ('SoyVitou/khmer-handwritten-dataset-4.2k', 'real_handwriting'),    ('SoyVitou/KhmerST-Dataset-OCR-Cropped', 'real_camera'),]for repo, name in SYNTH + REAL:    print('===', repo)    !cd {CODE} && python pack.py --repo {repo} --out /content/shards/{name} || echo "FAILED {repo}"

In [ ]:
# Copy shards to Drive so the next session doesn't re-download.!du -sh /content/shards/*!cp -r /content/shards/* {ROOT}/shards/print('shards backed up to Drive')

## 3. Phase 2 — CRNN baseline (~2.5h)Purpose is **not** final accuracy. It is a cheap correctness proof of the datapipeline. **Gate: val CER < 5%.** If CER is stuck above 15%, stop — that is alabel/charset bug, not a capacity problem. Go back to the audit.

In [ ]:
# W&B gives a live dashboard you can watch from your phone.# Paste the key at the prompt — never into a chat.import wandb; wandb.login()

In [ ]:
!cd {CODE} && python train.py \  --arch crnn \  --train-lmdb /content/shards/synth1m /content/shards/arial200k /content/shards/synth100k \               /content/shards/printed62k /content/shards/scene /content/shards/doclowres \  --val-lmdb /content/shards/real_handwriting \  --charset {ROOT}/audit/charset.json \  --out {ROOT}/checkpoints/crnn \  --epochs 10 --batch-size 256 --lr 3e-4 \  --samples-per-epoch 200000 \  --wandb khmer-ocr

## 4. Phase 3 — AttnOCR, the final model (20–30h across sessions)Attention decoder instead of CTC. CTC assumes a monotonic left-to-rightalignment; Khmer breaks that — subscripts stack vertically and some vowelsrender to the left of the consonant they follow in the encoding.This run **will** be interrupted by Colab. Just re-run this cell — it resumesfrom `last.pt` automatically.

In [ ]:
!cd {CODE} && python train.py \  --arch attn \  --train-lmdb /content/shards/synth1m /content/shards/arial200k /content/shards/synth100k \               /content/shards/printed62k /content/shards/scene /content/shards/doclowres \  --val-lmdb /content/shards/real_handwriting \  --charset {ROOT}/audit/charset.json \  --out {ROOT}/checkpoints/attn \  --epochs 20 --batch-size 128 --lr 7e-4 \  --samples-per-epoch 300000 \  --wandb khmer-ocr

## 5. Results — head-to-head on the held-out REAL data

In [ ]:
import pandas as pdfor name in ['crnn', 'attn']:    p = f'{ROOT}/checkpoints/{name}/progress.csv'    if os.path.exists(p):        df = pd.read_csv(p)        print(f'--- {name}: best CER {df.val_cer.min():.4f}  best WER {df.val_wer.min():.4f}')        display(df.tail(5))